# Quantize the merged model to GGUF (Layer 3, step 1)
Run on Colab. Needs the merged fp16 model from `merge_and_eval.ipynb` (it is only on Colab's disk, not on the Hub). If this is a fresh server, cell 3 re-creates it from your adapter.

Output: a ~2 GB `issuehawk-Q4_K_M.gguf`, uploaded to your Hub repo under `gguf/` so you can download it to your laptop.

## 1. Setup

In [ ]:
!git clone https://github.com/hannna24/Github-IssueHawk.git || (cd Github-IssueHawk && git pull)
%cd /content/Github-IssueHawk/backend/serving
!nvidia-smi -L

## 2. Log in to Hugging Face (Write token)

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass('Hugging Face token (write): '))

## 3. Make sure the merged model exists
If `../../checkpoints/merged` is missing (fresh server), run the merge cell below; otherwise skip it.

In [ ]:
!ls -la ../../checkpoints/merged

In [ ]:
# Only if the folder above is missing:
!pip install -q -U transformers peft accelerate
!pip uninstall -y -q torchao
%cd /content/Github-IssueHawk/training
!python merge_adapter.py --adapter hanabakeer24/issuehawk-qwen2.5-3b-lora --subfolder final-adapter --out ../checkpoints/merged
%cd /content/Github-IssueHawk/backend/serving

## 4. Convert to GGUF and quantize (Q4_K_M)
Builds llama.cpp, converts the model to an fp16 GGUF, then quantizes. About 5-10 minutes.

In [ ]:
!bash quantize.sh ../../checkpoints/merged ../../checkpoints/gguf Q4_K_M

## 5. Upload the quantized model to the Hub *(Colab's disk is wiped on disconnect)*
Only the small Q4_K_M file goes up, not the 6 GB fp16 intermediate.

In [ ]:
from huggingface_hub import HfApi
HfApi().upload_file(
    path_or_fileobj='../../checkpoints/gguf/issuehawk-Q4_K_M.gguf',
    path_in_repo='gguf/issuehawk-Q4_K_M.gguf',
    repo_id='hanabakeer24/issuehawk-qwen2.5-3b-lora',
)
print('uploaded')